In [3]:
%pip install pandas_datareader

Note: you may need to restart the kernel to use updated packages.


The system cannot find the path specified.


In [1]:
# -*- coding: utf-8 -*-
"""
Created on Fri Feb 13 12:08:09 2026

@Jinane
"""
import pandas as pd
import pandas_datareader.data as web
import datetime
import yfinance as yf

ModuleNotFoundError: No module named 'pandas_datareader'

In [ ]:
# U.S. Inflation (CPI) – Monetary Policy Regimes
# Dataset: Consumer Price Index (Monthly)
# Monthly since 1913
# 1300 observations
# Clear regime shifts (1970s inflation, 2008 crisis, COVID spike)
# Direct monetary policy implications
# Source: FRED (Federal Reserve Economic Data)
# Series ID: CPIAUCSL

start = datetime.datetime(1950, 1, 1)
end = datetime.datetime(2025, 1, 1)

cpi = web.DataReader('CPIAUCSL', 'fred', start, end)
cpi = cpi.dropna()

# convert to inflation rate
inflation = cpi.pct_change().dropna()
inflation.head()
inflation.to_csv('C:/Users/JOUNI/OneDrive - UNHCR/Documents/Records/data/test/inflation_series.csv')

# How You Can Classify
# Option A:
# High inflation regime
# Low inflation regime
# Option B:
# Pre-Volcker (pre-1980)
# Great Moderation (1985–2007)
# Post-COVID regime
# Your model can detect structural regime differences.

In [ ]:
###############################
# CO₂ Concentration – Climate Acceleration Detection
# Dataset: Mauna Loa CO₂ (Monthly)
# Monthly since 1958 (~800 observations)
# Strong trend + seasonality
# Policy relevance: climate change mitigation
# Tests model ability to detect deterministic trend vs structural shifts
# 🔹 Classification Idea
# Linear trend vs accelerating regime
# Pre-1990 vs Post-1990 acceleration
# Seasonal vs non-seasonal components
# Very interesting for structural dynamics.

url = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt"
co2 = pd.read_csv(url, delim_whitespace=True, comment='#',
                  names=["year","month","decimal","average","deseasonalized","days","std","unc"])

co2 = co2[co2["average"] > 0]
co2["date"] = pd.to_datetime(dict(year=co2.year, month=co2.month, day=1))
co2 = co2.set_index("date")

co2_series = co2["average"]
co2_series.head()

co2_series.to_csv('C:/Users/User/Documents/Records/data/test/co2_series.csv')


In [ ]:
# U.S. Unemployment Rate – Business Cycle Classification
# Dataset: UNRATE (Monthly)
# Monthly since 1948 (~900 obs)
# Clear recession spikes
# Direct macro policy implications
# Excellent for regime detection models
# Source : FRED
# Series ID: UNRATE
# Classification Strategy
# Recession vs expansion periods
# Pre vs post crisis structural change
# High-volatility vs stable periods
# You can align with NBER recession dates.

unrate = web.DataReader('UNRATE', 'fred', start, end)
unrate = unrate.dropna()
unrate.head()

unrate.to_csv('C:/Users/User/Documents/Records/data/test/unrate_series.csv')


# 🚀 Best For Your Paper?

# If your work is theoretical (record theory + ML classification):

# Goal	 ||                 Best Dataset
# Regime detection	        Unemployment
# Trend classification	    CO₂
# Structural breaks	        Inflation
# Policy discussion depth	Inflation + Unemployment

In [ ]:
# Define the ticker symbol for Gold (GLD is a popular Gold ETF)
ticker_symbol = "GLD"

# Create a ticker object
gold_ticker = yf.Ticker(ticker_symbol)

# Get historical market data (daily)
# period='max' for all available data, or use start and end dates
data = gold_ticker.history(period="1mo")

# Display the last few days
print(data.tail())

# Save to CSV
data.to_csv("daily_gold_prices.csv")


In [ ]:
# gold_daily_prices_and_metas.py
import os
import datetime as dt
import pandas as pd
import numpy as np
from pandas_datareader import data as pdr

# --------- Config ---------
YEARS_BACK = 10  # pull latest N years
AM_SERIES = "GOLDAMGBD228NLBM"  # LBMA Gold Price (AM) - USD/oz
PM_SERIES = "GOLDPMGBD228NLBM"  # LBMA Gold Price (PM) - USD/oz

# Optional: FRED API key for larger requests (set once in your environment)
# os.environ["FRED_API_KEY"] = "YOUR_FRED_API_KEY"

# --------- Date range ---------
end = dt.date.today()
start = end - dt.timedelta(days=365 * YEARS_BACK + 30)  # pad a bit

# --------- Download from FRED ---------
am = pdr.DataReader(AM_SERIES, "fred", start, end)
pm = pdr.DataReader(PM_SERIES, "fred", start, end)

am.columns = ["gold_usd_am"]
pm.columns = ["gold_usd_pm"]

# --------- Merge & basic cleaning ---------
df = am.join(pm, how="outer")
df.index.name = "date"

# Spot proxy: prefer PM fix (closer to daily close), fallback to AM if PM is NaN
df["gold_usd_fix"] = df["gold_usd_pm"].combine_first(df["gold_usd_am"])

# Drop rows with no price at all
df = df.dropna(subset=["gold_usd_fix"])

# --------- Feature engineering ("other metas") ---------
# Returns
df["ret_d"] = df["gold_usd_fix"].pct_change()

# Rolling volatility (annualized; 21 ~ 1M trading days, 252 ~ 1Y)
def ann_vol(x, window):
    return x.rolling(window).std() * np.sqrt(252)

df["vol_21"]  = ann_vol(df["ret_d"], 21)
df["vol_252"] = ann_vol(df["ret_d"], 252)

# Rolling mean & z-score (63 ~ quarter)
df["ma_63"] = df["gold_usd_fix"].rolling(63).mean()
df["sd_63"] = df["gold_usd_fix"].rolling(63).std()
df["z_63"]  = (df["gold_usd_fix"] - df["ma_63"]) / df["sd_63"]

# Running max & drawdown
df["running_max"] = df["gold_usd_fix"].cummax()
df["drawdown"]    = df["gold_usd_fix"] / df["running_max"] - 1.0

# Calendar features
df["year"]      = df.index.year
df["month"]     = df.index.month
df["weekday"]   = df.index.weekday  # 0=Mon
df["month_end"] = df.index.is_month_end

# Reorder columns nicely
base_cols = ["gold_usd_am", "gold_usd_pm", "gold_usd_fix"]
feat_cols = ["ret_d", "vol_21", "vol_252", "ma_63", "z_63", "running_max", "drawdown",
             "year", "month", "weekday", "month_end"]
df = df[base_cols + feat_cols]

# --------- Save outputs ---------
df[base_cols].to_csv("gold_fix_am_pm.csv", float_format="%.6f")
df.to_csv("gold_features.csv", float_format="%.6f")

print("Saved: gold_fix_am_pm.csv, gold_features.csv")
print(df.tail(10))